In [ ]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
import string

## Dataset preparation 
1. Load the email and keep only the relevant features
2. Load the Smishing dataset: since it lacks headers, we dynamically assign columns (v1,v2) and rename them for standardization
3.  Merge both datasets to build a more robust
4.  Function execution 

In [ ]:
def prepare_datasets():
    df_emails = pd.read_csv('spam_ham_dataset.csv')
    df_emails = df_emails[['label', 'text']].copy()
    df_emails.rename(columns={'text': 'original_text'}, inplace=True)
    df_emails['source'] = 'email'

    df_sms = pd.read_csv('spamhamdata.csv', sep='\t', header=None, names=['v1', 'v2'], encoding='latin-1')
    df_sms.rename(columns={'v1': 'label', 'v2': 'original_text'}, inplace=True)
    df_sms['source'] = 'sms'

    df_merged = pd.concat([df_emails, df_sms], ignore_index=True)
    return df_merged

In [ ]:
df_base = prepare_datasets()
print(f"Total initial records: {len(df_base)}")

## Data Cleansing: Handling missing values and duplicates
1. We drop duplicate records based on the message content, keeping only the first occurrence. This is crucial to prevent model overfitting, ensuring the algorithm learns patterns rather than memorizing repeated spam compaigns
2. We remove rows where the original_text is empty (NaN)
3. Using the subset parameter ensures we only drop rows with nulls or duplicates in this critical text column, preserving the data point if non-essential metadata is missing
4. Execution of the cleaning function

In [ ]:
def clean_duplicates_nulls(df):
    initial_count = len(df)

    df_clean = df.drop_duplicates(subset=['original_text'], keep='first').copy()
    df_clean = df_clean.dropna(subset=['original_text'])

    final_count = len(df_clean)
    print(f"Removed {initial_count - final_count} duplicate or null records.")
    return df_clean

In [ ]:
df_unique = clean_duplicates_nulls(df_base)
print(f"Total records after cleaning: {len(df_unique)}")

## Creating a checkpoint and structure verification
1. Local backup: We save the current progress to a CSV file (dataset_processed_phase1.csv). This acts as a reliable checkpoint; if the notebook kernel crashes or restarts in subsequent steps, we can load this file without rerunning the initial ingestion and cleaning processes
2. We use the index = False parameter to prevent Pandas from exporting the row indices as an unnamed "garbage" column
3. Verification: We print the dataset's structural information (info()) to confirm the absence of null values and check data types. Finally, we output a sample (head()) for a quick visual sanity check

In [ ]:
df_unique.to_csv('data_processed_phase1.csv', index=False)
print("Processed data saved to 'data_processed_phase1.csv'\n")
print(df_unique.info())
print("\n ---First 5 records---")
print(df_unique.head())